# Data Mining - TP Règles d'association

### Marwan BENLKHIR - ING4 Data&IA APP Grp01

In [74]:
import matplotlib.pyplot as plt
import numpy as np
from numpy import where
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth
import time

##### 1. Formulation du problème

- Voici un extrait des données qui représente un première colonne le numéro de la transaction (panier d'achat) et en deuxième colonne la liste de produit du panier.

- L'entreprise souhaite améliorer la taille du panier par transaction pour augmenter son chiffre d'affaire. Pour réaliser leur objectif, nous procédons à une fouille de donnée afin d'obtenir des règles d'association. Ensuite les décideurs auront les connaissances pour faire des campagnes de promotions ou réorganiser l'agencement des produits dans leurs magasins afin d'inciter les clients à l'achat.

- Exemple :
    - Un client qui achète du beurre a 60% de probabilité d'acheter du pain. {butter}->{bread}
    - {pasta}->{tomato}

In [75]:
df = pd.read_csv("transactions_promos.csv")
df.head()

,transaction_id,items_space_separated
0,1,bread butter milk eggs
1,2,bread butter jam
2,3,bread milk cereal
3,4,milk cereal banana
4,5,bread cheese tomato


##### 2. Application de l'algorithme Apriori

In [76]:
setlist = set()
for row in df['items_space_separated'] :
    for i in row.split(' ') :
        setlist.add(i)
df1 = pd.DataFrame()
for product in setlist:
    df1[product] = df['items_space_separated'].str.contains(product)
df1 = df1.fillna(0).astype(int)
print(df1.head())

   honey  banana  bananas  soy_sauce  cheese  cereal  juice  beans  \
0      0       0        0          0       0       0      0      0   
1      0       0        0          0       0       0      0      0   
2      0       0        0          0       0       1      0      0   
3      0       1        0          0       0       1      0      0   
4      0       0        0          0       1       0      0      0   

   peanut_butter  oats  ...  tomato  oranges  strawberries  olive_oil  fish  \
0              0     0  ...       0        0             0          0     0   
1              0     0  ...       0        0             0          0     0   
2              0     0  ...       0        0             0          0     0   
3              0     0  ...       0        0             0          0     0   
4              0     0  ...       1        0             0          0     0   

   jam  eggs  noodles  tomato_sauce  croissant  
0    0     1        0             0          0  
1    1

Le support minimal de l'algorithme a été configuré à 10% car 20% est trop haut pour obtenir des règles d'association.

In [77]:
start = time.time()
frequent_itemsets = apriori(df1, min_support=0.1, use_colnames=True)
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.6)
end = time.time()
#print(rules.columns)
rules2 = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift','representativity']]
#frequentItems = pd.DataFrame()
#frequentItems['frequentItems'] = rules2.apply(lambda x : ', '.join(list(x['antecedents']) + list(x['consequents'])), axis=1)
#frequentItems['representativity'] = rules2['representativity']
print('Apriori Frequent Items : \n',frequent_itemsets.sort_values(by=['support'], ascending=False).head(),'\n\n')

print('Apriori Association rules : \n',rules2.sort_values(by=['confidence'], ascending=False).head())

Apriori Frequent Items : 
     support  itemsets
3     0.225    (milk)
4     0.225   (bread)
7     0.200  (butter)
11    0.200  (tomato)
0     0.125  (banana) 


Apriori Association rules : 
               antecedents     consequents  support  confidence  lift  \
2          (tomato_sauce)         (pasta)      0.1         1.0   8.0   
4          (tomato_sauce)        (tomato)      0.1         1.0   5.0   
5  (tomato, tomato_sauce)         (pasta)      0.1         1.0   8.0   
6         (tomato, pasta)  (tomato_sauce)      0.1         1.0  10.0   
7   (tomato_sauce, pasta)        (tomato)      0.1         1.0   5.0   

   representativity  
2               1.0  
4               1.0  
5               1.0  
6               1.0  
7               1.0  


/opt/miniconda3/envs/venvMacOS/lib/python3.13/site-packages/mlxtend/frequent_patterns/fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


Plusieurs règles d'association déterminées à partir du dataset ont une confidence de 1, c'est-à-dire qu'à chaque fois qu'un panier contient de la sauce tomate, il contient aussi des pâtes, par exemple. Le trio tomate, sauce tomate et pâte est toujours acheté en même temps.

Exemple : 
- {tomato_sauce}->{tomato}, confidence : 100%, support : 10%

- {tomato, pasta}->{tomato_sauce} confidence : 100%, support : 10%

##### 3. Comparaison avec l'algorithme FP-Growth

In [78]:
start2 = time.time()
frequent_itemsets2 = fpgrowth(df1, min_support=0.1, use_colnames=True)
rules3 = association_rules(frequent_itemsets2, metric='confidence', min_threshold=0.6)
end2 = time.time()
rules4 = rules3[['antecedents', 'consequents', 'support', 'confidence', 'lift','representativity']]
print('FPGrowth Frequent Items : \n',frequent_itemsets2.sort_values(by=['support'], ascending=False).head(),'\n\n')
print('FPGrowth Association rules : \n',rules4.sort_values(by=['confidence'], ascending=False).head())

FPGrowth Frequent Items : 
    support  itemsets
0    0.225   (bread)
1    0.225    (milk)
2    0.200  (butter)
5    0.200  (tomato)
4    0.125  (banana) 


FPGrowth Association rules : 
               antecedents     consequents  support  confidence  lift  \
2          (tomato_sauce)         (pasta)      0.1         1.0   8.0   
4          (tomato_sauce)        (tomato)      0.1         1.0   5.0   
5  (tomato, tomato_sauce)         (pasta)      0.1         1.0   8.0   
6         (tomato, pasta)  (tomato_sauce)      0.1         1.0  10.0   
7   (tomato_sauce, pasta)        (tomato)      0.1         1.0   5.0   

   representativity  
2               1.0  
4               1.0  
5               1.0  
6               1.0  
7               1.0  


/opt/miniconda3/envs/venvMacOS/lib/python3.13/site-packages/mlxtend/frequent_patterns/fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [79]:
print('Number of rules : Apriori ', len(rules2), ', FPGrowth ',len(rules4))
print('Execution time : Apriori ', start-end, ', FPGrowth ',start2-end2)

Number of rules : Apriori  10 , FPGrowth  10
Execution time : Apriori  -0.008411884307861328 , FPGrowth  -0.00559687614440918


Les deux algorithmes ont à peu près les mêmes résultats sur ce jeu de données mais FPGrowth à un temps d'execution un peu plus court. J'imagine qu'avec un grand nombre de données les différences de résultats et de temps d'execution se fera remarquer.
FPGrowth n'a pas besoin de générer tous les candidats en générant un arbre contrairement à Apriori. Le FP-tree et le point fort de son algorithme et permet à FPGrowth de facroriser les calculs, de réduire son espace en mémoire. FPGrowth est scalable et a favoriser pour les datasets volumineux.

##### 4. Interprétation et recommandations

In [80]:
print('FPGrowth Association rules : \n',rules4.sort_values(by=['lift'], ascending=False).head())

FPGrowth Association rules : 
               antecedents      consequents  support  confidence  lift  \
6         (tomato, pasta)   (tomato_sauce)      0.1         1.0  10.0   
8          (tomato_sauce)  (tomato, pasta)      0.1         1.0  10.0   
2          (tomato_sauce)          (pasta)      0.1         1.0   8.0   
3                 (pasta)   (tomato_sauce)      0.1         0.8   8.0   
5  (tomato, tomato_sauce)          (pasta)      0.1         1.0   8.0   

   representativity  
6               1.0  
8               1.0  
2               1.0  
3               1.0  
5               1.0  


La valeur de lift nous indique à quel point A influence l'achat de B ou si c'est juste une coincidence. Ici, lift vaut 10 pour l'achat de sauce tomate quand des tomates et des pâtes sont dans le panier et nous indique que cette règle a un lien très fort. On trouve que la sauce tomate, les tomates et les pâtes sont liés de manière très forte dans toutes les directions.

La stratégie à mettre en place serait de placer ces trois produits précédents à plusieurs mètres d'écart de façon à ce que le client passe devant d'autres articles qu'il serait tenté d'acheter pendant le trajet entre la sauce tomate, les tomates et les pâtes car nous savons qu'il achètera à coup sûr qu'à partir d'un, il ira chercher les autres. Aussi, le pain est l'article le plus vendu. Alors nous pouvons proposer une réduction sur celui-ci pour augmenter davantage le nombre de vente de ce produit. Le pain est quoi qu'il arrive très acheté, avec un promotion les clients en consommeront en plus grand quantité encore.

Il est surprenant qu'avec des tomates et des pâtes dans le panier, un client achète à 100% de confidence de la sauce tomate. On pourrait se dire que cela fait doublon entre les tomates et la sauce tomate. Aussi, le deuxième article le plus vendu est le lait alors qu'il n'est dans aucune règle d'association avec une confidence supérieur à 60%.